《Think Stats》 3판은 [Bookshop.org](https://bookshop.org/a/98697/9781098190255)와 [Amazon](https://amzn.to/42lmxwu)에서 구할 수 있습니다. 두 링크는 원저자의 제휴 링크입니다. 책이 도움이 되었다면 Green Tea Press의 [무료 공개 교육 자료 제작을 후원](https://greenteapress.com/support/?book=ts3)하는 것도 생각해 주세요.

# 생존분석

**생존분석**은 어떤 상태가 얼마나 오래 지속되는지 설명하는 방법입니다. 사람의 수명뿐 아니라 기계·전자 부품의 수명, 어떤 사건까지 걸리는 시간, 경우에 따라 거리에도 적용할 수 있습니다.

먼저 전구의 수명을 살펴보고, 미국의 첫 결혼 연령이 여러 세대에 걸쳐 어떻게 달라졌는지 분석하겠습니다.

[원문 노트북을 Colab에서 실행하기](https://colab.research.google.com/github/AllenDowney/ThinkStats/blob/30ecbdcf05723f1190cae2aeaff5848f59f8ca26/nb/chap13.ipynb).

In [1]:
from os.path import basename, exists


def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + local)


download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/thinkstats.py")

In [2]:
try:
    import empiricaldist
except ImportError:
    %pip install empiricaldist

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from thinkstats import decorate

## 생존함수

**생존함수**는 지속시간이 주어진 시점보다 길 확률을 나타내는 함수입니다. 2007년의 전구 실험을 예로 보겠습니다. 연구진은 새 전구 50개를 계속 켜 두고 12시간마다 확인했습니다. 그 사이 꺼진 전구의 수명을 기록하고 50개가 모두 꺼질 때까지 실험을 계속했습니다. 자료를 가져오는 코드는 노트북에 있습니다.

자료는 Menon, V. J.와 Agrawal, D. C.(2007), 「Renewal Rate of Filament Lamps: Theory and Experiment」, *Journal of Failure Analysis and Prevention*, 421쪽 표 2에서 왔습니다. [DOI: 10.1007/s11668-007-9074-9](https://doi.org/10.1007/s11668-007-9074-9). 전사 자료와 설명은 [이 저장소](https://gist.github.com/epogrebnyak/7933e16c0ad215742c4c104be4fbdeb1)에 있습니다.

In [4]:
download(
    "https://gist.github.com/epogrebnyak/7933e16c0ad215742c4c104be4fbdeb1/raw/c932bc5b6aa6317770c4cbf43eb591511fec08f9/lamps.csv"
)

자료를 다음과 같이 읽습니다.

In [5]:
df = pd.read_csv("lamps.csv", index_col=0)
df.tail()

`h`는 시간(hour) 단위의 수명이고 `f`는 그 시각에 수명이 끝난 전구 수입니다. `Pmf`에 넣고 정규화해 수명 분포를 만듭니다.

In [6]:
from empiricaldist import Pmf

pmf_bulblife = Pmf(df["f"].values, index=df["h"])
pmf_bulblife.normalize()

`make_cdf`로 각 시각까지 수명이 끝난 비율을 구합니다. 예를 들어 1656시간 이하에서 전체 전구의 78%가 꺼졌습니다.

In [7]:
cdf_bulblife = pmf_bulblife.make_cdf()
cdf_bulblife[1656]

생존함수는 각 시각 **이후**까지 켜져 있는 비율입니다. 따라서 1에서 CDF의 값을 빼면 됩니다.

$$S(t)=P(T>t)=1-F(t)$$

In [8]:
complementary_cdf = 1 - cdf_bulblife
complementary_cdf[1656]

1656시간보다 오래 켜진 전구는 22%입니다. `empiricaldist`의 `Surv`가 생존함수를 나타내고 `make_surv`가 이를 계산합니다.

In [9]:
surv_bulblife = cdf_bulblife.make_surv()
surv_bulblife[1656]

CDF와 생존함수를 함께 그리면 모든 시각에서 두 값의 합이 1임을 볼 수 있습니다.

In [10]:
cdf_bulblife.plot(ls="--", label="CDF")
surv_bulblife.plot(label="Survival")

decorate(xlabel="Light bulb duration (hours)", ylabel="Probability")

하나를 알면 다른 하나를 구할 수 있다는 점에서 두 함수는 같은 정보를 담습니다. 다만 생존분석에서는 생존곡선을 자주 쓰며, 다음 개념인 위험함수를 구하는 데에도 유용합니다.

## 위험함수

이 자료의 `h`는 그 시각에 끝나는 12시간 구간을 뜻합니다. 어떤 전구가 그 구간 시작까지 살아남았다면, 그 구간에서 꺼질 확률은 얼마일까요?

구간 뒤에도 살아남은 비율 `surv`와 그 구간에 꺼진 비율 `pmf`를 더하면, 구간 시작에 아직 켜져 있어 사건을 겪을 수 있었던 **위험집단**의 비율이 됩니다. 1656시간 구간에서는 전체의 26%입니다.

In [11]:
at_risk = pmf_bulblife + surv_bulblife
at_risk[1656]

그 구간에서 꺼진 전구는 전체의 4%입니다.

In [12]:
pmf_bulblife[1656]

**위험**은 해당 구간의 사건 비율을 위험집단 비율로 나눈 조건부 확률입니다.

In [13]:
hazard = pmf_bulblife / at_risk
hazard[1656]

따라서 그 구간 시작까지 살아남은 전구의 약 15%가 구간 안에서 꺼졌습니다. `empiricaldist`의 `make_hazard`로도 같은 `Hazard` 객체를 만들 수 있습니다.

In [14]:
hazard_bulblife = surv_bulblife.make_hazard()
hazard_bulblife[1656]

위험함수를 그립니다.

In [15]:
hazard_bulblife.plot()
decorate(xlabel="Light bulb duration (hours)", ylabel="Hazard")

구간마다 값이 들쭉날쭉합니다. 특히 위험집단이 작은 꼬리에서는 이 그림만으로 패턴을 읽기 어렵습니다. 위험의 누적합인 **누적위험함수**를 그리는 편이 도움이 됩니다.

In [16]:
cumulative_hazard = hazard_bulblife.cumsum()
cumulative_hazard.plot()

decorate(xlabel="Light bulb duration (hours)", ylabel="Cumulative hazard")

누적위험의 기울기가 가파르면 그 구간의 위험이 높고, 평평하면 낮습니다. 이 그림에서는 1500~2000시간 부근이 가파릅니다. 그 뒤의 모양은 유난히 오래 살아남은 전구 한 개에도 크게 좌우되므로 다른 실험에서는 달라질 수 있습니다.

> 번역자 주: 여기서 위험은 한 구간의 조건부 확률입니다. 연속시간의 순간 위험률과는 단위가 다릅니다. 이산시간에서는 $S(t)=\prod_{u\le t}(1-h(u))$이며, 위험의 단순 누적합을 언제나 $-\log S(t)$와 같다고 놓을 수는 없습니다.

이 개념을 더 큰 자료에 적용하겠습니다.

## 결혼 자료

여러 나라에서 결혼이 늦어지고 미혼으로 남는 비율도 변하고 있습니다. 미국의 변화를 NSFG 응답자 자료와 생존분석으로 살펴보겠습니다.

앞 장의 임신 파일은 임신 한 건이 행 하나였습니다. 이번 응답자 파일은 응답자 한 사람이 행 하나입니다. 원저자는 1982~2019년 아홉 차례 조사에서 결혼 관련 자료를 모았습니다. 내려받는 코드는 노트북에 있습니다.

원저자가 여러 조사 주기를 합쳐 만든 CSV입니다. 정리 과정은 [자료 준비 노트북](https://github.com/AllenDowney/MarriageNSFG/blob/master/clean_nsfg.ipynb)에 있습니다.

In [17]:
filename = "marriage_nsfg_female.csv.gz"
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/" + filename)

CSV를 읽습니다.

In [18]:
resp = pd.read_csv("marriage_nsfg_female.csv.gz")
resp.shape

7만 명이 넘는 응답자의 자료에 다음 변수가 있습니다.

- `cmbirth`: 생년월. 모든 응답자에게 알려져 있습니다.
- `cmintvw`: 면접 연월. 모든 응답자에게 알려져 있습니다.
- `cmmarrhx`: 해당하고 알려진 경우 첫 결혼 연월.
- `evrmarry`: 면접 전 결혼한 적이 있으면 1, 없으면 0.

날짜는 1899년 12월 이후 지난 개월 수인 century-month로 저장되어 있습니다. 1은 1900년 1월입니다. 출생 세대별 변화를 보기 위해 출생 연대를 계산하겠습니다. `//`는 정수 나눗셈으로 몫을 내림합니다.

In [19]:
month0 = pd.to_datetime("1899-12-31")


def decade_of_birth(cmbirth):
    date = month0 + pd.DateOffset(months=cmbirth)
    return date.year // 10 * 10

각 사람의 출생 연대를 `cohort` 열에 넣습니다. **코호트**는 출생 연대처럼 공통 특성을 가져 함께 분석하는 집단입니다. 코호트별 인원도 확인합니다.

In [20]:
from thinkstats import value_counts

resp["cohort"] = resp["cmbirth"].apply(decade_of_birth)
value_counts(resp["cohort"])

1950~1980년대생은 각 연대에 1만 명 이상이며, 더 이르거나 늦은 연대는 적습니다. 결혼 당시 나이와 면접 당시 나이를 계산하겠습니다.

In [21]:
resp["agemarr"] = (resp["cmmarrhx"] - resp["cmbirth"]) / 12
resp["age"] = (resp["cmintvw"] - resp["cmbirth"]) / 12

다음 함수는 자료와 코호트 목록을 받아, 코호트마다 첫 결혼 나이의 `Surv`를 만듭니다. `dropna=False`는 아직 결혼하지 않아 결혼 나이가 없는 사례도 분모에 포함합니다. 다만 이것만으로 중도절단을 올바르게 처리한 것은 아닙니다.

In [22]:
from empiricaldist import Surv


def make_survival_map(resp, cohorts):
    surv_map = {}

    grouped = resp.groupby("cohort")
    for cohort in cohorts:
        group = grouped.get_group(cohort)
        surv_map[cohort] = Surv.from_seq(group["agemarr"], dropna=False)

    return surv_map

함수를 호출합니다.

In [23]:
cohorts = [1980, 1960, 1940]
surv_map = make_survival_map(resp, cohorts)

1940·1960·1980년대생의 결과를 그립니다.

In [24]:
import matplotlib.pyplot as plt
import cycler

# Extract the default color cycle
default_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

# Define the desired line styles
linestyles = ["--", "-", "-.", "--", "-", "-.", "--", "-", "-.", "--"]

# Ensure we cycle through styles and colors properly
custom_cycler = cycler.cycler(color=default_colors) + cycler.cycler(
    linestyle=linestyles
)

# Apply the new cycler
plt.rc("axes", prop_cycle=custom_cycler)

In [25]:
for cohort, surv in surv_map.items():
    surv.plot(label=f"{cohort}s")

ylim = [-0.05, 1.05]
decorate(xlabel="Age", ylabel="Prob never married", ylim=ylim)

그대로 읽으면 앞 세대일수록 결혼이 이르고 결국 결혼하는 비율도 높아 보입니다. 하지만 아직 해석하면 안 됩니다. 두 문제가 있습니다.

- NSFG는 일부 집단을 의도적으로 과대표집합니다.
- 아직 결혼하지 않은 사람의 불완전한 관측을 제대로 처리하지 않았습니다.

첫 문제에는 **가중 부트스트랩**, 둘째에는 **카플란–마이어 추정**을 사용하겠습니다.

## 가중 부트스트랩

`finalwgt`는 각 응답자의 표집 가중치로, 그 응답자가 모집단에서 대표하는 인원 수를 나타냅니다. 가중치에 비례해 행을 다시 뽑으면 과대표집을 조정하는 데 사용할 수 있습니다. 다음 함수는 자료와 가중치 열 이름을 받아 가중 복원추출을 합니다.

In [26]:
def resample_rows_weighted(df, column="finalwgt"):
    n = len(df)
    weights = df[column]
    return df.sample(n, weights=weights, replace=True)

여러 조사 주기가 섞여 있으므로 주기별로 따로 재표집한 다음 다시 합칩니다.

In [27]:
def resample_cycles(resp):
    grouped = resp.groupby("cycle")
    samples = [resample_rows_weighted(group) for _, group in grouped]
    return pd.concat(samples)

먼저 한 번 재표집하겠습니다.

In [28]:
# Seed the random number generator so we get the same results every time
np.random.seed(1)

In [29]:
sample = resample_cycles(resp)

뒤에서는 이를 여러 번 반복해 표집 변동을 보겠습니다. 다음 그림은 가중 재표집 결과와 앞 절의 무보정 결과인 점선을 비교합니다.

In [30]:
for label, surv in surv_map.items():
    surv.plot(ls=":", color="gray", alpha=0.6)

survs_resampled = make_survival_map(sample, cohorts)

for label, surv in survs_resampled.items():
    surv.plot(label=label)

decorate(xlabel="Age", ylabel="Prob never married", ylim=ylim)

In [31]:
plt.rc("axes", prop_cycle=plt.rcParamsDefault["axes.prop_cycle"])

차이가 상당합니다. 과대표집을 고려해야 함을 보여 줍니다. 이번에는 아직 사건을 관측하지 못한 자료를 처리하겠습니다.

## 위험함수 추정하기

전구 실험에서는 50개의 수명이 모두 끝나 전체 지속시간을 알았습니다. 결혼 자료에서는 면접 전에 결혼한 사람의 나이는 알지만, 아직 미혼인 사람이 언제 결혼할지, 결혼할지 자체는 모릅니다.

이런 관측을 **중도절단**(censoring)이라고 합니다. 정보를 일부러 숨긴다는 뜻이 아니라 관측이 먼저 끝났다는 뜻입니다. 그래도 미혼 응답자가 결혼한다면 면접 당시 나이보다 뒤라는 부분 정보가 있습니다.

이 정보를 위험집단 계산에 넣어 위험함수를 추정한 뒤 생존함수로 바꾸는 것이 카플란–마이어 방법입니다. 가중 재표집 자료에서 한 코호트를 고르겠습니다.

In [32]:
resp60 = sample.query("cohort == 1960")

면접 전에 결혼한 사람에게서는 첫 결혼 나이를 고릅니다. 원문의 이 재표집에서는 9921명이며 완료 사례라고 부릅니다.

In [33]:
complete = resp60.query("evrmarry == 1")["agemarr"]
complete.count()

면접 당시 미혼인 사람에게서는 면접 나이를 고릅니다. 원문에서는 5468명이며 진행 중인 사례라고 부릅니다.

In [34]:
ongoing = resp60.query("evrmarry == 0")["age"]
ongoing.count()

각 나이 직전까지 미혼 상태로 관측된 사람을 위험집단에 포함해야 합니다. 완료·진행 사례의 시간을 각각 `FreqTab`으로 세겠습니다.

In [35]:
from empiricaldist import FreqTab

ft_complete = FreqTab.from_seq(complete)
ft_ongoing = FreqTab.from_seq(ongoing)

예를 들어 이 재표집에서 25세에 처음 결혼한 응답자는 58명입니다.

In [36]:
ft_complete[25]

25세에 면접을 받았고 그때까지 미혼인 응답자는 5명입니다.

In [37]:
ft_ongoing[25]

정규화하지 않은 `Surv`를 만들면 각 나이 이후까지 이어진 완료·진행 사례의 수를 얻습니다.

In [38]:
surv_complete = ft_complete.make_surv()
surv_ongoing = ft_ongoing.make_surv()

25세 이후에 처음 결혼한 사람은 2848명입니다.

In [39]:
surv_complete[25]

25세 이후 면접에서도 미혼인 사람은 2273명입니다.

In [40]:
surv_ongoing[25]

네 수를 더하면 25세 직전 위험집단입니다. 의학에서 비롯된 ‘위험’이라는 말이 결혼에는 어색할 수 있지만, 여기서는 해당 나이 직전까지 첫 결혼을 하지 않아, 그 나이에 첫 결혼을 할 수 있는 사람을 뜻합니다.

In [41]:
at_risk = ft_complete[25] + ft_ongoing[25] + surv_complete[25] + surv_ongoing[25]
at_risk

위험집단 중 25세에 결혼한 수는 `ft_complete[25]`이므로 다음 비율로 위험을 구합니다.

In [42]:
hazard = ft_complete[25] / at_risk
hazard

이번에는 모든 나이에서 같은 계산을 하겠습니다. 두 도수표의 나이 인덱스를 `union`으로 합칩니다.

In [43]:
ts = pd.Index.union(ft_complete.index, ft_ongoing.index)

합친 나이들에서 완료·진행 도수와 그 이후 생존 사례 수를 조회해 위험집단 수를 구합니다.

In [44]:
at_risk = ft_complete(ts) + ft_ongoing(ts) + surv_complete(ts) + surv_ongoing(ts)

각 나이의 사건 수를 위험집단 수로 나누어 `Hazard`에 담습니다.

In [45]:
from empiricaldist import Hazard

hs = ft_complete(ts) / at_risk
hazard = Hazard(hs, ts)

누적위험함수를 그립니다.

In [46]:
hazard.cumsum().plot()

decorate(xlabel="Age", ylabel="Cumulative hazard")

20~30세 구간의 기울기가 가장 큽니다. 그때까지 미혼인 사람의 조건부 결혼 확률이 이 연령대에서 높다는 뜻입니다. 이후 기울기는 완만해집니다.

## 생존함수 추정하기

반대로 위험함수에서 생존함수를 구할 수도 있습니다. 위험은 그 나이까지 미혼인 사람이 그 나이에 결혼할 조건부 확률입니다. 그 여사건은 해당 구간을 미혼으로 넘길 확률입니다.

나이 $t$ 뒤까지 미혼이려면 그때까지 모든 구간을 미혼으로 지나야 하므로, 조건부 생존확률을 곱합니다.

In [47]:
ps = (1 - hazard).cumprod()

`Hazard.make_surv`가 이 곱을 계산합니다.

In [48]:
surv = hazard.make_surv()

가중치는 고려했지만 중도절단을 처리하지 않은 앞의 점선과 비교합니다.

In [49]:
survs_resampled[1960].plot(ls=":", color="gray", label="resampled")
surv.plot(label="Kaplan-Meier")

decorate(xlabel="Age", ylabel="Prob never married", ylim=ylim)

중도절단을 올바르게 다루는 것이 얼마나 중요한지 보입니다.

이런 생존곡선과 관련해 잘못 알려진 사례가 있습니다. 1986년 《Newsweek》는 미혼인 40세 여성은 결혼할 확률보다 테러로 목숨을 잃을 확률이 높다는 선정적인 주장을 보도했습니다. 잘못된 분석에 근거한 주장이었고, 이미 진행 중이던 사회 변화도 반영하지 못했습니다. 매체는 2006년에 오류를 인정하는 기사를 실었습니다.

이 사례의 분석과 사회적 반응을 살펴보길 권합니다. 통계를 주의 깊게 계산하고, 결과를 비판적으로 해석하고, 정확하고 정직하게 전달할 책임을 생각하게 합니다.

배경은 [Snopes의 해설](https://www.snopes.com/fact-check/marry-go-round/)을 참고하세요.

다음 함수는 완료 사례의 시간과 진행 사례의 관측 종료 시간을 받아 위험함수를 추정하는 과정을 묶습니다.

In [50]:
def estimate_hazard(complete, ongoing):
    """Kaplan-Meier estimation."""
    ft_complete = FreqTab.from_seq(complete)
    ft_ongoing = FreqTab.from_seq(ongoing)

    surv_complete = ft_complete.make_surv()
    surv_ongoing = ft_ongoing.make_surv()

    ts = pd.Index.union(ft_complete.index, ft_ongoing.index)
    at_risk = ft_complete(ts) + ft_ongoing(ts) + surv_complete(ts) + surv_ongoing(ts)

    hs = ft_complete(ts) / at_risk
    return Hazard(hs, ts)

코호트의 자료에서 두 종류의 시간을 꺼내 위험을 추정하고 생존함수로 바꾸는 함수입니다.

In [51]:
def estimate_survival(group):
    """Estimate the survival function."""
    complete = group.query("evrmarry == 1")["agemarr"]
    ongoing = group.query("evrmarry == 0")["age"]
    hf = estimate_hazard(complete, ongoing)
    sf = hf.make_surv()
    return sf

곧 이 함수들로 여러 생존곡선을 계산하겠습니다. 먼저 다른 구현과 비교하겠습니다.

> 번역자 주: 중도절단 뒤의 결과를 추정하려면, 분석에서 고려한 조건 아래 관측 종료가 사건 발생시간에 관한 별도 정보를 주지 않는다는 가정이 필요합니다. 마지막 추적 이후의 생존율은 관측 자료만으로 알 수 없습니다. 같은 시각의 사건과 중도절단은 위험집단에 함께 포함한 뒤 사건을 계산하는 관례를 사용합니다.

## Lifelines

Python의 `lifelines` 패키지에는 카플란–마이어 추정 등 생존분석 도구가 있습니다.

내려받는 노트북에서는 필요한 경우 `lifelines`를 설치합니다. 이 블로그의 짧은 실행창에서는 위험집단과 사건 수를 직접 계산합니다.

In [52]:
try:
    import lifelines
except ImportError:
    %pip install lifelines

앞에서 작성한 `estimate_survival`로 먼저 생존함수를 구합니다.

In [53]:
surv = estimate_survival(resp60)

이제 `lifelines`가 요구하는 지속시간과 사건 관측 여부 형식으로 자료를 바꿉니다.

In [54]:
complete = complete.dropna()
durations = np.concatenate([complete, ongoing])
event_observed = np.concatenate([np.ones(len(complete)), np.zeros(len(ongoing))])

`KaplanMeierFitter`를 만들고 적합합니다.

In [55]:
from lifelines import KaplanMeierFitter

kmf = KaplanMeierFitter()
kmf.fit(durations=durations, event_observed=event_observed)

`plot`은 생존함수와 구간을 함께 표시합니다. 다만 이 재표집 자료에 기본 구간을 바로 적용하는 것만으로 원래 복합표본 조사의 불확실성을 제대로 계산한 것은 아닙니다.

In [56]:
kmf.plot()

decorate(xlabel="Age", ylabel="Prob never married", ylim=ylim)

`lifelines`의 곡선은 시간축 0에서 시작하며 그때 생존확률은 1입니다. 그 뒤의 점추정값들은 앞에서 직접 계산한 값과 부동소수점 오차 범위에서 같습니다.

In [57]:
ps = kmf.survival_function_["KM_estimate"].drop(0)
np.allclose(ps, surv)

다음에는 과대표집을 고려한 가중 재표집을 여러 번 수행해 변동을 살펴보겠습니다.

## 신뢰구간

지금의 카플란–마이어 곡선은 재표집 한 번의 결과입니다. 표집 변동을 보려면 여러 번 재표집하고 매번 곡선을 추정할 수 있습니다.

다음 함수는 코호트별 생존함수를 사전으로 만듭니다. 앞의 `make_survival_map`과 구조는 같지만, 중도절단을 처리하지 않는 `Surv.from_seq` 대신 `estimate_survival`을 사용합니다.

In [58]:
def estimate_survival_map(resp, cohorts):
    """Make a dictionary from cohorts to Surv objects."""
    surv_map = {}

    grouped = resp.groupby("cohort")
    for cohort in cohorts:
        group = grouped.get_group(cohort)
        surv_map[cohort] = estimate_survival(group)

    return surv_map

101번 가중 재표집해, 코호트별 생존함수 사전 101개를 만듭니다.

In [59]:
cohorts = [1940, 1950, 1960, 1970, 1980, 1990]

surv_maps = [estimate_survival_map(resample_cycles(resp), cohorts) for i in range(101)]

사전들에서 특정 코호트의 곡선을 꺼내 반투명하게 겹쳐 그립니다.

In [60]:
def plot_cohort(surv_maps, cohort, color):
    """Plot results for a single cohort."""
    survs = [surv_map[cohort] for surv_map in surv_maps]
    for surv in survs:
        surv.plot(color=color, alpha=0.05)

    x, y = surv.index[-1], surv.iloc[-1]
    plt.text(x + 1, y, f"{cohort}s", ha="left", va="center")

1940~1990년대 출생 코호트들을 비교합니다.

In [61]:
colors = [f"C{i}" for i in range(len(cohorts))]

for cohort, color in zip(cohorts, colors):
    plot_cohort(surv_maps, cohort, color)

xlim = [8, 55]
decorate(xlabel="Age", ylabel="Prob never married", xlim=xlim, ylim=ylim)

탐색에는 충분하지만 선들이 흐려지고 표기가 겹쳐, 출판용 그림이라면 더 다듬을 필요가 있습니다. 다음 패턴을 볼 수 있습니다.

- 1940년대생이 가장 일찍 결혼했습니다. 1950·1960년대생은 더 늦게 결혼했지만 관측한 높은 연령에서의 미혼 비율은 비슷합니다.
- 1970년대생은 결혼이 늦어졌고, 관측 연령에서 미혼 비율도 높아졌습니다.
- 1980·1990년대생은 더 늦게 결혼하는 패턴을 보입니다. 다만 앞으로 변할 수 있습니다.

젊은 코호트의 이후 경향을 알려면 후속 조사 자료가 더 필요합니다.

> 번역자 주: 젊은 코호트의 마지막 관측값을 ‘평생 결혼하지 않을 비율’로 읽으면 안 됩니다. 가중 재표집 곡선의 퍼짐도 조사 층화·군집을 완전히 재현한 공식 설계기반 신뢰구간과는 구별해야 합니다.

## 기대 잔여시간

전체 지속시간의 분포가 있으면, 이미 일정 시간이 지난 뒤 남은 시간이 평균적으로 얼마인지 구할 수 있습니다. NSFG 임신 기간 분포로 남은 시간을 계산해 보겠습니다.

1장에서와 같은 NSFG 임신 자료를 사용합니다.

In [62]:
download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/nsfg.py")
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dct")
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dat.gz")

In [63]:
try:
    import statadict
except ImportError:
    %pip install statadict

자료를 읽고 첫아이와 다른 아이로 나눕니다.

In [64]:
from nsfg import get_nsfg_groups

live, firsts, others = get_nsfg_groups()

먼저 가중 재표집을 한 번 하겠습니다.

In [65]:
sample = resample_rows_weighted(live, "finalwgt")

임신 기간의 PMF를 그립니다.

In [66]:
pmf_durations = Pmf.from_seq(sample["prglngth"])

임신 36주가 시작되었다고 하겠습니다. 가장 흔한 기간이 39주이므로 3~4주가 남았을 것이라 예상할 수 있습니다. 정확히 계산하려고 전체 기간이 36주 이상인 값만 고릅니다.

In [67]:
t = 36
is_remaining = pmf_durations.qs >= t

선택한 값에서 36을 빼 현재를 0으로 옮기고 새 PMF를 만듭니다.

In [68]:
ps = pmf_durations.ps[is_remaining]
qs = pmf_durations.qs[is_remaining] - t

pmf_remaining = Pmf(ps, qs)

일부 값만 골랐으므로 확률 합이 1보다 작습니다. 다시 정규화합니다.

In [69]:
pmf_remaining.normalize()

이것이 36주 시작 시점의 남은 시간 분포입니다.

In [70]:
pmf_remaining.bar(label="Week 36")
decorate(xlabel="Remaining duration (weeks)", ylabel="PMF")

그 평균이 기대 잔여시간입니다.

In [71]:
pmf_remaining.mean()

다음 함수는 주어진 시점 `t`에서 남은 시간의 PMF를 계산합니다.

In [72]:
def compute_pmf_remaining(pmf, t):
    """Distribution of remaining time."""
    is_remaining = pmf.qs >= t
    ps = pmf.ps[is_remaining]
    qs = pmf.qs[is_remaining] - t
    pmf_remaining = Pmf(ps, qs)
    pmf_remaining.normalize()
    return pmf_remaining

이를 36~43주 각각에서 평가해 기대 잔여시간을 구합니다.

In [73]:
def expected_remaining(pmf):
    index = range(36, 44)
    expected = pd.Series(index=index)

    for t in index:
        pmf_remaining = compute_pmf_remaining(pmf, t)
        expected[t] = pmf_remaining.mean()

    return expected

재표집 한 번의 결과를 그립니다.

In [74]:
expected = expected_remaining(pmf_durations)
expected

표집 변동을 보기 위해 여러 번 재표집해 같은 계산을 반복합니다.

In [75]:
for i in range(21):
    sample = resample_rows_weighted(live, "finalwgt")
    pmf_durations = Pmf.from_seq(sample["prglngth"])
    expected = expected_remaining(pmf_durations)
    expected.plot(color="C0", alpha=0.1)

decorate(
    xlabel="Weeks of pregnancy", ylabel="Expected remaining time (weeks)", ylim=[0, 3.4]
)

36~39주에는 남은 시간의 평균이 줄어들고 39주 시작에서는 약 0.6주입니다. 그런데 그 뒤에는 거의 평평합니다. 40주 시작의 잔여시간도 약 0.6주이고, 41~43주에도 비슷합니다. 초조하게 출산을 기다리는 사람에게는 뜻밖의 모양일 수 있습니다.

조건이 매번 달라지기 때문입니다. 40주에도 진행 중인 임신만 남기면, 이미 끝난 사례들이 빠진 다른 조건부 분포를 보게 됩니다. 이 계산은 조사 자료의 기술적인 조건부 평균이며 개인의 분만 시점을 정하는 예측이나 의료 판단 기준은 아닙니다.

## 용어 정리

- **생존분석**: 관심 사건까지 걸리는 시간을 다루는 방법들.
- **생존함수**: $t$보다 오래 지속될 확률 $P(T>t)$.
- **위험함수**: 그때까지 사건 없이 남은 사례 중 해당 구간에 사건이 생기는 비율.
- **누적위험**: 이 장의 구간별 위험을 누적한 값.
- **가중 부트스트랩**: 표집 가중치를 사용해 복원추출하는 재표집 방법.
- **중도절단**: 사건이 아직 없거나 관측되지 않아 지속시간을 부분적으로만 아는 상태.
- **카플란–마이어 추정**: 위험집단과 관측 사건을 사용해 중도절단이 있는 생존함수를 추정하는 방법.
- **코호트**: 출생 연대 같은 공통 특성을 가진 분석 집단.

## 연습문제

### 연습문제 13.1

첫 결혼의 지속시간을 분석해 보세요. 사건은 별거나 사망이 아니라 이혼으로 한정합니다. `cmdivorcx`는 첫 결혼의 이혼 연월이며 century-month 단위입니다.

- 완료 사례: 이혼 연월과 첫 결혼 연월이 모두 있으면 두 값의 차이를 구합니다.
- 진행 사례: 한 번만 결혼했고 현재도 결혼 상태인 사람을 고릅니다. `fmarno`는 결혼 횟수, `rmarital=1`은 현재 결혼 상태입니다. 면접까지 지난 기간을 구합니다.

날짜가 근사 기록인 사례에서는 작은 음수가 나올 수 있지만 1년을 넘지는 않아야 합니다. 위험함수와 생존함수를 추정하고 누적위험을 그리세요. 어느 기간에 이혼의 조건부 위험이 높은가요? 관측 범위에서 이혼의 누적비율은 얼마인가요?

> 번역자 주: 이 정의가 포함하지 않은 별거·사망·재혼 사례의 제외가 선택편향을 만들 수 있습니다. 단순한 곡선의 마지막 값만으로 모든 결혼의 평생 이혼 확률을 확정하지 마세요.

In [76]:
# I suggest you use a single resampling of the data

sample = resample_cycles(resp)

In [77]:
# Solution goes here

In [78]:
# Solution goes here

In [79]:
# Solution goes here

In [80]:
# Solution goes here

In [81]:
# Solution goes here

In [82]:
# Solution goes here

In [83]:
# Solution goes here

In [84]:
# Solution goes here

### 연습문제 13.2

2012년 남캘리포니아대 연구진은 1800년대와 1900년대 초 스웨덴 출생 코호트의 수명을 추정했습니다. 각 나이까지 살아남은 사람 중 그 나이에 사망하는 비율은 이 장의 위험함수에 해당합니다. 원저자는 논문 그림을 온라인 도구로 수치화해 CSV로 저장했습니다.

자료 출처: Beltrán-Sánchez, Crimmins, Finch(2012), 「Early cohort mortality predicts the rate of aging in the cohort: a historical analysis」, *Journal of Developmental Origins of Health and Disease* 3(5), 380–386. 자료를 가져오는 코드는 노트북에 있습니다.

In [85]:
download(
    "https://github.com/AllenDowney/ThinkStats/raw/v3/data/mortality_rates_beltran2012.csv"
)

표를 다음과 같이 읽습니다.

In [86]:
mortality = pd.read_csv("mortality_rates_beltran2012.csv", header=[0, 1]).dropna()

다음 함수는 로그 사망률을 보간해 0~99세의 위험함수를 근사합니다. 원그림의 관측 나이를 벗어난 부분은 외삽이라는 점에 유의하세요.

In [87]:
from scipy.interpolate import interp1d
from empiricaldist import Hazard


def make_hazard(ages, rates):
    """Make a Hazard function by interpolating a Series.

    series: Series

    returns: Hazard
    """
    interp = interp1d(ages, rates, fill_value="extrapolate")
    xs = np.arange(0, 100)
    ys = np.exp(interp(xs))
    return Hazard(ys, xs)

`Hazard` 객체를 만듭니다.

In [88]:
ages = mortality["1800", "X"].values
rates = mortality["1800", "Y"].values
hazard = make_hazard(ages, rates)

사망률을 그립니다.

In [89]:
hazard.plot()

decorate(xlabel="Age (years)", ylabel="Hazard")

`make_surv`로 생존함수를, `make_cdf`로 CDF를 구해 그리세요. `make_pmf`로 수명 분포를 만들고, `compute_pmf_remaining`으로 나이별 평균 잔여수명을 구해 그리세요.

유아기에는 나이가 들수록 오히려 잔여수명이 증가하는 패턴이 보일 수 있습니다. 영아 사망률이 매우 높았기 때문에 그 시기를 넘긴 아이의 조건부 전망이 달라지는 것입니다. 약 5세 이후에는 나이가 적은 사람이 더 오래 살 것으로 기대하는 익숙한 모양으로 돌아옵니다. 원저자의 《Probably Overthinking It》 5장에도 이런 역설적인 통계 사례들이 나옵니다.

책의 안내는 [probablyoverthinking.it](https://probablyoverthinking.it)에 있습니다.

In [90]:
# Solution goes here

In [91]:
# Solution goes here

In [92]:
# Solution goes here

In [93]:
# Solution goes here

In [94]:
# Solution goes here

[Think Stats: Exploratory Data Analysis in Python, 3rd Edition](https://allendowney.github.io/ThinkStats/index.html)

Copyright 2024 [Allen B. Downey](https://allendowney.com)

코드 라이선스: [MIT](https://mit-license.org/)

본문·그림과 한국어 번역 라이선스: [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)